# Persistent Lyra-2 Custom Trajectory Inference

Use this notebook when you SSH into the machine and want to keep Lyra/DA3/T5 weights loaded in VRAM between runs. Start Jupyter from the repo root, for example:

```bash
cd /home/dx/dong/lyra/Lyra-2
PYTHONPATH=. jupyter lab --no-browser --ip=127.0.0.1 --port=8888
```

Then connect through an SSH tunnel from your laptop. The weights remain loaded as long as this notebook kernel stays alive. Run the final cleanup cell before closing the session if you want to release VRAM without restarting the kernel.

In [ ]:
# Setup: run once after starting the kernel.
from pathlib import Path
import gc
import json
import os
import sys
from types import SimpleNamespace

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

ROOT = Path("/home/dx/dong/lyra/Lyra-2").resolve()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import cv2
import numpy as np
import torch
import torch.nn.functional as F

from lyra_2._ext.imaginaire.utils import log, misc
from lyra_2._ext.imaginaire.visualize.video import save_img_or_video
from lyra_2._src.inference.lyra2_ar_inference import run_lyra2_sample, safe_to
from lyra_2._src.inference.lyra2_custom_traj_inference import load_trajectory
from lyra_2._src.inference.lyra2_zoomgs_inference import _da3_infer_depth_intrinsics_single
from lyra_2._src.utils.model_loader import load_model_from_checkpoint

torch.enable_grad(False)
torch.backends.cudnn.enabled = False
print(f"Repo: {ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Configuration: edit these paths/values between runs as needed.
args = SimpleNamespace(
    experiment="lyra2",
    checkpoint_dir="checkpoints/model",
    output_path="outputs/notebook_custom_traj",
    guidance=5.0,
    shift=5.0,
    num_sampling_step=35,
    seed=1,
    fps=16,
    num_frames=481,  # must be 1 + 80k for the default lyra2 experiment
    pose_scale=1.1,
    resolution="480,832",
    context_parallel_size=1,
    lora_paths=None,
    lora_weights=None,
    offload=False,
    offload_when_prompt=False,
    prompt="",
    prompt_dir=None,
    captions_path="assets/custom_trajectory_examples/escroom2_navigation/captions.json",
    prompt_suffix="",
    depth_backend="da3",
    da3_model_name="depth-anything/DA3NESTED-GIANT-LARGE-1.1",
    da3_model_path_custom=None,
    da3_frame_interval=8,
    da3_max_history_frames=10,
    da3_include_ar_chunk_last_frames=False,
    da3_use_predicted_pose=False,
    da3_predicted_pose_continuation=False,
    use_moge_scale=True,
    use_dmd=False,
    use_dmd_scheduler=False,
    ablate_same_t5=False,
    warp_chunk_size=None,
    num_retrieval_views=1,
    disable_cache_update=False,
    multiview_ids=None,
    offload_da3_diffusion=False,
)

input_image_path = "assets/custom_trajectory_examples/escroom2_navigation/first_frame.png"
trajectory_path = "assets/custom_trajectory_examples/escroom2_navigation/trajectory.npz"
video_name = "escroom2_navigation"

DMD_LORA_PATH = "checkpoints/lora/dmd_distillation.safetensors"
if args.use_dmd:
    args.use_dmd_scheduler = True
    args.lora_paths = list(args.lora_paths or []) + [DMD_LORA_PATH]
    args.lora_weights = list(args.lora_weights or []) + [1.0]

Path(args.output_path).mkdir(parents=True, exist_ok=True)

In [ ]:
# Load weights: run once per kernel. Re-running this cell will not reload if model is already present.
if "model" in globals() and model is not None:
    print("Lyra model is already loaded; keeping existing VRAM allocation.")
else:
    misc.set_random_seed(seed=args.seed, by_rank=True)

    negative_prompt_data = torch.load(
        "checkpoints/text_encoder/negative_prompt.pt",
        map_location="cpu",
        weights_only=False,
    )

    experiment_opts = [
        "model.config.use_mp_policy_fsdp=False",
        "model.config.keep_original_net_dtype=False",
    ]
    if args.lora_paths:
        experiment_opts += ["model.config.net.postpone_checkpoint=True"]

    model, config = load_model_from_checkpoint(
        config_file="lyra_2/_src/configs/config.py",
        experiment_name=args.experiment,
        checkpoint_path=args.checkpoint_dir,
        enable_fsdp=False,
        instantiate_ema=False,
        load_ema_to_reg=False,
        experiment_opts=experiment_opts,
    )

    if args.lora_paths:
        lora_names = []
        for lora_path in args.lora_paths:
            lora_names.append(model.load_lora_weights(lora_path))
        model.set_weights_and_activate_adapters(lora_names, args.lora_weights)
        if hasattr(model, "net") and hasattr(model.net, "enable_selective_checkpoint"):
            model.net.enable_selective_checkpoint(model.net.sac_config, model.net.blocks)

    desired_dtype = model.tensor_kwargs.get("dtype", None)
    desired_device = model.tensor_kwargs.get("device", "cuda" if torch.cuda.is_available() else "cpu")
    if desired_dtype is not None:
        model.net = model.net.to(device=desired_device, dtype=desired_dtype)
    model.eval()

    from lyra_2._src.inference.depth_utils import load_da3_model, load_moge_model
    da3_model = load_da3_model(
        da3_model_name=args.da3_model_name,
        da3_model_path_custom=args.da3_model_path_custom,
        device=desired_device,
    )
    da3_model.eval()

    moge_model = None
    if args.use_moge_scale:
        moge_model = load_moge_model(desired_device)
        moge_model.eval()

    print("Loaded Lyra, DA3", "+ MoGe" if moge_model is not None else "")

if torch.cuda.is_available():
    print(torch.cuda.memory_summary(abbreviated=True))

In [ ]:
# Helper functions. These reuse the already-loaded model objects above.
def _make_t5_inputs(base_name: str, num_frames: int):
    from lyra_2._src.inference.get_t5_emb import get_umt5_embedding, get_umt5_embedding_offloaded

    desired_dtype = model.tensor_kwargs.get("dtype", None)
    desired_device = model.tensor_kwargs.get("device", "cuda" if torch.cuda.is_available() else "cpu")
    neg_t5 = misc.to(negative_prompt_data["t5_text_embeddings"], **model.tensor_kwargs)

    captions_file = None
    if args.captions_path:
        cap_path = Path(args.captions_path)
        captions_file = cap_path / f"{base_name}.json" if cap_path.is_dir() else cap_path
        if not captions_file.is_file():
            captions_file = None

    def embed_caption(text: str):
        if args.prompt_suffix:
            text = text.rstrip() + " " + args.prompt_suffix
        if args.offload_when_prompt:
            emb = get_umt5_embedding_offloaded(text, device=desired_device).to(dtype=desired_dtype)
        else:
            emb = get_umt5_embedding(text, device=desired_device).to(dtype=desired_dtype)
        return emb

    extra = {}
    if captions_file is not None:
        captions_dict = json.loads(captions_file.read_text())
        chunk_keys_int = [k for k in sorted(int(k) for k in captions_dict) if k < num_frames]
        if len(chunk_keys_int) > 1:
            chunk_embs = []
            chunk_masks = []
            for ck in chunk_keys_int:
                emb = embed_caption(captions_dict[str(ck)])
                if emb.dim() == 3:
                    emb = emb[0]
                S, D = emb.shape
                S = min(S, 512)
                D = min(D, 4096)
                padded_emb = torch.zeros(512, 4096, dtype=desired_dtype, device=desired_device)
                padded_emb[:S, :D] = emb[:S, :D]
                padded_mask = torch.zeros(512, dtype=desired_dtype, device=desired_device)
                padded_mask[:S] = 1.0
                chunk_embs.append(padded_emb)
                chunk_masks.append(padded_mask)
            extra["t5_chunk_keys"] = torch.tensor(chunk_keys_int, dtype=torch.long, device=desired_device).unsqueeze(0)
            extra["t5_chunk_embeddings"] = torch.stack(chunk_embs).unsqueeze(0)
            extra["t5_chunk_mask"] = torch.stack(chunk_masks).unsqueeze(0)
            extra["sample_frame_indices"] = torch.arange(num_frames, dtype=torch.long, device=desired_device).unsqueeze(0)
            return extra["t5_chunk_embeddings"][:, 0], neg_t5, extra
        if chunk_keys_int:
            caption = captions_dict[str(chunk_keys_int[0])]
        else:
            caption = args.prompt
    elif args.prompt:
        caption = args.prompt
    elif args.prompt_dir:
        caption = (Path(args.prompt_dir) / f"{base_name}.txt").read_text().strip()
    else:
        raise RuntimeError("Set args.captions_path, args.prompt, or args.prompt_dir before running generation.")

    t5 = embed_caption(caption)
    if t5.dim() == 2:
        t5 = t5.unsqueeze(0)
    elif t5.dim() == 3 and t5.shape[0] != 1:
        t5 = t5[:1]
    return t5, neg_t5, extra


def generate_custom_trajectory(image_path: str, traj_path: str, name: str | None = None):
    desired_device = model.tensor_kwargs.get("device", "cuda" if torch.cuda.is_available() else "cpu")
    target_h, target_w = [int(x) for x in args.resolution.split(",")]
    num_frames = int(args.num_frames)
    image_path = Path(image_path)
    traj_path = Path(traj_path)
    base_name = name or image_path.stem
    output_path = Path(args.output_path) / f"{base_name}.mp4"

    misc.set_random_seed(seed=args.seed, by_rank=True)
    w2cs_T_44, Ks_T_33 = load_trajectory(str(traj_path), num_frames, target_hw=(target_h, target_w), pose_scale=args.pose_scale)

    bgr = cv2.imread(str(image_path))
    if bgr is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    rgb_t = torch.from_numpy(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))

    image_chw01, depth_hw, _K_33_da3, mask_hw = _da3_infer_depth_intrinsics_single(
        da3_model=da3_model,
        img_rgb_uint8=rgb_t,
        target_hw=(target_h, target_w),
    )

    if args.use_moge_scale and moge_model is not None:
        from lyra_2._src.inference.depth_utils import moge_infer_depth_intrinsics
        moge_model.to(desired_device)
        with torch.nn.attention.sdpa_kernel([torch.nn.attention.SDPBackend.MATH]):
            _, moge_depth_hw, _, moge_mask_hw = moge_infer_depth_intrinsics(
                moge_model, rgb_t, depth_pred_hw=(target_h, target_w), target_hw=(target_h, target_w)
            )
        da3_d = depth_hw.to(moge_depth_hw.device)
        da3_m = mask_hw.to(moge_mask_hw.device)
        valid_mask = (da3_m > 0.5) & (moge_mask_hw > 0.5)
        if valid_mask.sum() > 10:
            inv_da3 = 1.0 / (da3_d[valid_mask] + 1e-6)
            inv_moge = 1.0 / (moge_depth_hw[valid_mask] + 1e-6)
            denom = (inv_da3 * inv_da3).sum()
            if denom > 1e-8:
                scale = (inv_da3 * inv_moge).sum() / denom
                if scale > 1e-6:
                    depth_hw = depth_hw / scale.to(depth_hw.device)
        moge_model.cpu()
        del moge_depth_hw, moge_mask_hw, da3_d, da3_m
        torch.cuda.empty_cache()

    img_bchw = image_chw01.to(device=desired_device) * 2.0 - 1.0
    H, W = image_chw01.shape[-2:]
    t5, neg_t5, extra_t5 = _make_t5_inputs(base_name, num_frames)

    data_batch = {
        "video": img_bchw.unsqueeze(2),
        "t5_text_embeddings": t5,
        "neg_t5_text_embeddings": neg_t5,
        "fps": torch.tensor([args.fps], dtype=torch.int32, device=desired_device),
        "padding_mask": torch.zeros((1, 1, H, W), dtype=model.tensor_kwargs["dtype"], device=desired_device),
        "is_preprocessed": torch.tensor([True], dtype=torch.bool, device=desired_device),
        "camera_w2c": w2cs_T_44.unsqueeze(0).to(dtype=torch.float32, device=desired_device),
        "intrinsics": Ks_T_33.unsqueeze(0).to(dtype=torch.float32, device=desired_device),
        "depth": depth_hw.unsqueeze(0).unsqueeze(0).repeat(1, num_frames, 1, 1).to(device=desired_device),
        **extra_t5,
    }

    skip_keys = {"camera_w2c", "intrinsics", "depth", "t5_chunk_keys", "sample_frame_indices"}
    data_batch = safe_to(data_batch, device=desired_device, dtype=model.tensor_kwargs.get("dtype", None), skip_keys=skip_keys)
    result = run_lyra2_sample(model, data_batch, args, da3_model=da3_model, show_progress=True, log_prefix=f"{base_name}_notebook")

    video_01 = (result["video"][0].clamp(-1, 1) * 0.5 + 0.5).float().cpu()
    save_img_or_video(video_01, str(output_path.with_suffix("")), fps=args.fps)
    del result, data_batch, video_01
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return output_path

In [ ]:
# Run generation. You can edit the config/path cell above and run this cell repeatedly.
video_path = generate_custom_trajectory(input_image_path, trajectory_path, video_name)
print(f"Saved: {video_path}")
if torch.cuda.is_available():
    print(torch.cuda.memory_summary(abbreviated=True))

## Unload And Final Cleanup

Run this cell when you are done and want to release VRAM while keeping the notebook server alive.

In [ ]:
# Final cleanup: explicitly drop loaded models, T5 globals, and CUDA cache.
def unload_all_models():
    global model, config, da3_model, moge_model, negative_prompt_data
    for name in ["model", "config", "da3_model", "moge_model", "negative_prompt_data"]:
        if name in globals():
            globals()[name] = None

    try:
        import lyra_2._src.inference.get_t5_emb as t5_emb
        t5_emb.t5_encoder = None
        t5_emb._t5_offloaded = None
    except Exception as exc:
        print(f"T5 cleanup skipped: {exc}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
        print(torch.cuda.memory_summary(abbreviated=True))

unload_all_models()